# Worldle Final Project

This notebook builds a playable Worldle-style country guessing game using `wdo`, `ipyleaflet`, and `ipywidgets`.

## wdo functions completed

- `bbox_from_feature`
- `make_map`
- `add_geojson`
- `fit_map_to_geojson`
- `choose_target`
- `feature_center`
- `guess_feedback`
- `format_feedback`

## Known bugs / limitations

- Country centers use bounding-box centers, so large countries like Russia and the United States may have imperfect arrows.
- Some flag names may not match perfectly because country polygons use ISO-3 codes while flags use ISO-2 codes.
- This is designed to run inside Jupyter/Codespaces, not as a standalone web app.

## Cell 1 — Import Libraries

This cell imports the Python libraries, mapping tools, widgets, display tools, and custom `wdo` functions needed for the game.

In [24]:
%reload_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import json
import base64
import random

# Find the main project root by walking upward until we find Resources/wdo
CURRENT = Path.cwd()
PROJECT_ROOT = CURRENT

while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "Resources" / "wdo").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

WDO_PARENT = PROJECT_ROOT / "Resources"

if not (WDO_PARENT / "wdo").exists():
    raise FileNotFoundError(
        f"Could not find Resources/wdo. Current folder is {CURRENT}. "
        f"Checked upward and ended at {PROJECT_ROOT}."
    )

# Add Resources to Python path so import wdo works
if str(WDO_PARENT) not in sys.path:
    sys.path.insert(0, str(WDO_PARENT))

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

from wdo.io.geojson_tools import load_geojson, iter_features
from wdo.maps.leaflet_helpers import make_map, add_geojson, fit_map_to_geojson
from wdo.games.worldle import choose_target, guess_feedback, format_feedback

import wdo

print("Current folder:", CURRENT)
print("Project root:", PROJECT_ROOT)
print("Using wdo from:", wdo.__file__)

Current folder: /workspaces/spacial_mapping/completed_assignments/04-Worldle
Project root: /workspaces/spacial_mapping
Using wdo from: /workspaces/spacial_mapping/Resources/wdo/__init__.py


# Cell 2 - Load Country GeoJSON Data

In [25]:
possible_country_paths = [
    PROJECT_ROOT / "Resources" / "Data" / "countries_export.json",
    PROJECT_ROOT / "4543-5993-Spatial-Data" / "Resources" / "Data" / "countries_export.json",
]

countries_path = None

for path in possible_country_paths:
    if path.exists():
        countries_path = path
        break

if countries_path is None:
    raise FileNotFoundError("Could not find countries_export.json")

with open(countries_path, "r", encoding="utf-8") as f:
    countries_geojson = json.load(f)

print("Countries path:", countries_path)
print("Loaded object type:", type(countries_geojson))

if isinstance(countries_geojson, dict):
    if countries_geojson.get("type") == "FeatureCollection":
        features = countries_geojson.get("features", [])
    else:
        features = countries_geojson.get("features", [])

elif isinstance(countries_geojson, list):
    if len(countries_geojson) > 0 and isinstance(countries_geojson[0], dict):
        if countries_geojson[0].get("type") == "Feature":
            features = countries_geojson
        elif countries_geojson[0].get("type") == "FeatureCollection":
            features = countries_geojson[0].get("features", [])
        else:
            features = countries_geojson
    else:
        features = []

else:
    raise TypeError("Unknown countries file format")

print("Feature count:", len(features))

if len(features) == 0:
    raise ValueError("The countries file loaded, but no usable features were found.")

print("First country:", features[0]["properties"].get("ADMIN"))
print("First ISO3:", features[0]["properties"].get("ISO_A3"))
print("First geometry type:", features[0]["geometry"].get("type"))

countries_feature_collection = {
    "type": "FeatureCollection",
    "features": features,
}

Countries path: /workspaces/spacial_mapping/4543-5993-Spatial-Data/Resources/Data/countries_export.json
Loaded object type: <class 'list'>
Feature count: 255
First country: Aruba
First ISO3: ABW
First geometry type: Polygon


# Cell 3 - Load FLags

In [17]:
possible_flag_paths = [
    PROJECT_ROOT / "Resources" / "Data" / "flag-icons" / "country.json",
    PROJECT_ROOT / "4543-5993-Spatial-Data" / "Resources" / "Data" / "flag-icons" / "country.json",
]

flag_index_path = None

for path in possible_flag_paths:
    if path.exists():
        flag_index_path = path
        break

if flag_index_path is None:
    print("Flag index not found. Game will still work without flag images.")
    flag_index = {}
else:
    with open(flag_index_path, "r", encoding="utf-8") as f:
        flag_index = json.load(f)

print("Flag index path:", flag_index_path)

if flag_index:
    print("Flag index type:", type(flag_index))
    
    if isinstance(flag_index, dict):
        first_key = list(flag_index.keys())[0]
        print("Example flag key:", first_key)
        print("Example flag value:", flag_index[first_key])
    
    elif isinstance(flag_index, list):
        print("First flag item:", flag_index[0])

Flag index path: /workspaces/spacial_mapping/Resources/Data/flag-icons/country.json
Flag index type: <class 'list'>
First flag item: {'capital': 'Kabul', 'code': 'af', 'continent': 'Asia', 'flag_1x1': 'flags/1x1/af.svg', 'flag_4x3': 'flags/4x3/af.svg', 'iso': True, 'name': 'Afghanistan'}


# Cell 4 - Country Lookup

In [18]:
ALIASES = {
    "United States of America": "United States",
    "Russia": "Russian Federation",
    "South Korea": "Korea, Republic of",
    "North Korea": "Korea, Democratic People's Republic of",
    "Iran": "Iran, Islamic Republic of",
    "Syria": "Syrian Arab Republic",
    "Vietnam": "Viet Nam",
    "Laos": "Lao People's Democratic Republic",
    "Tanzania": "Tanzania, United Republic of",
    "Venezuela": "Venezuela, Bolivarian Republic of",
    "Bolivia": "Bolivia, Plurinational State of",
    "Moldova": "Moldova, Republic of",
    "Brunei": "Brunei Darussalam",
}


def build_country_lookup(features, flag_index):
    """
    Build iso3 -> name, iso2, flag_path lookup.
    Handles flag_index as either a dict or a list.
    """
    name_to_iso2 = {}

    if isinstance(flag_index, dict):
        for iso2, info in flag_index.items():
            if isinstance(info, dict):
                name = (
                    info.get("name")
                    or info.get("country")
                    or info.get("title")
                    or info.get("label")
                )
            else:
                name = str(info)

            if name:
                name_to_iso2[name.lower()] = iso2.lower()

    elif isinstance(flag_index, list):
        for item in flag_index:
            if not isinstance(item, dict):
                continue

            iso2 = (
                item.get("iso2")
                or item.get("code")
                or item.get("alpha2")
                or item.get("ISO_A2")
            )

            name = (
                item.get("name")
                or item.get("country")
                or item.get("title")
                or item.get("label")
            )

            if iso2 and name:
                name_to_iso2[name.lower()] = iso2.lower()

    lookup = {}
    misses = []

    for feature in features:
        props = feature["properties"]
        name = props.get("ADMIN")
        iso3 = props.get("ISO_A3")

        search_name = ALIASES.get(name, name)
        iso2 = name_to_iso2.get(str(search_name).lower())

        if iso2 and flag_index_path is not None:
            flag_path = flag_index_path.parent / "flags" / "4x3" / f"{iso2}.svg"
        else:
            flag_path = None
            misses.append(name)

        lookup[iso3] = {
            "name": name,
            "iso2": iso2,
            "flag_path": flag_path,
        }

    print("Country lookup count:", len(lookup))
    print("Flag misses:", len(misses))
    print("First 20 misses:", misses[:20])

    return lookup


country_lookup = build_country_lookup(features, flag_index)

# Quick check
first_iso3 = features[0]["properties"]["ISO_A3"]
print("Example lookup:", first_iso3, country_lookup[first_iso3])

Country lookup count: 239
Flag misses: 48
First 20 misses: ['Aland', 'Ashmore and Cartier Islands', 'French Southern and Antarctic Lands', 'Bajo Nuevo Bank (Petrel Is.)', 'Saint Barthelemy', 'The Bahamas', 'Bolivia', 'Ivory Coast', 'Cyprus No Mans Area', 'Republic of Congo', 'Coral Sea Islands', 'Cape Verde', 'Northern Cyprus', 'Dhekelia Sovereign Base Area', 'Guinea Bissau', 'Hong Kong S.A.R.', 'Indian Ocean Territories', 'Iran', 'Siachen Glacier', 'Baykonur Cosmodrome']
Example lookup: ABW {'name': 'Aruba', 'iso2': 'aw', 'flag_path': PosixPath('/workspaces/spacial_mapping/Resources/Data/flag-icons/flags/4x3/aw.svg')}


# Cell 5 - Display helper function

In [19]:
def svg_to_data_uri(path):
    """
    Convert an SVG flag file into a data URI for HTML display.
    """
    if path is None:
        return None

    path = Path(path)

    if not path.exists():
        return None

    with open(path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")

    return f"data:image/svg+xml;base64,{encoded}"


def proximity_style(distance_km):
    """
    Return color and emoji based on distance.
    """
    if distance_km < 500:
        return "#2a9d8f", "🔥"
    elif distance_km < 2000:
        return "#e9c46a", "🌡️"
    else:
        return "#e76f51", "🧊"


def render_guess_row(country_name, flag_path, arrow, distance_km, correct=False):
    """
    Return HTML for one guess row.
    Shows flag, country name, arrow, distance, and hot/cold feedback.
    """
    flag_uri = svg_to_data_uri(flag_path)

    if flag_uri:
        flag_html = f'<img src="{flag_uri}" width="45" style="border:1px solid #ccc;">'
    else:
        flag_html = "🏳️"

    if correct:
        distance_text = "Correct!"
        color = "#2a9d8f"
        emoji = "🎉"
    else:
        distance_text = f"{distance_km:,.0f} km"
        color, emoji = proximity_style(distance_km)

    return f"""
    <div style="
        display: grid;
        grid-template-columns: 60px 180px 60px 120px 60px;
        align-items: center;
        gap: 10px;
        border-left: 8px solid {color};
        background-color: #f8f8f8;
        padding: 10px;
        margin: 6px 0;
        border-radius: 8px;
        font-size: 16px;
    ">
        <div>{flag_html}</div>
        <div><b>{country_name}</b></div>
        <div style="font-size: 22px;">{arrow}</div>
        <div style="color:{color}; font-weight:bold;">{distance_text}</div>
        <div style="font-size: 22px;">{emoji}</div>
    </div>
    """

# Cell 6b - Test Guess Row Display

In [20]:
sample_iso3 = features[0]["properties"]["ISO_A3"]
sample_info = country_lookup[sample_iso3]

display(HTML(render_guess_row(
    country_name=sample_info["name"],
    flag_path=sample_info["flag_path"],
    arrow="↗",
    distance_km=1234,
    correct=False
)))

# Cell 7 - Game Class

In [21]:
class WorldleGame:
    """
    Small class to keep the Worldle game state organized.
    """

    def __init__(self, features, lookup, seed=None, max_guesses=6):
        self.features = features
        self.lookup = lookup
        self.target = choose_target(features, seed=seed)
        self.target_iso3 = self.target["properties"]["ISO_A3"]
        self.target_name = self.target["properties"]["ADMIN"]
        self.guesses = []
        self.max_guesses = max_guesses
        self.finished = False

    def submit_guess(self, guess_iso3):
        """
        Submit a country guess using its ISO-3 code.
        Returns feedback from wdo.games.worldle.guess_feedback.
        """
        if self.finished:
            return None

        guess_feature = None

        for feature in self.features:
            if feature["properties"]["ISO_A3"] == guess_iso3:
                guess_feature = feature
                break

        if guess_feature is None:
            raise ValueError("Invalid guess ISO3 code")

        result = guess_feedback(guess_feature, self.target)
        self.guesses.append(result)

        if result["correct"]:
            self.finished = True
            result["message"] = "win"
        elif len(self.guesses) >= self.max_guesses:
            self.finished = True
            result["message"] = "loss"
        else:
            result["message"] = "continue"

        return result

# Cell 7b - Test Game Initialization

In [22]:
test_game = WorldleGame(features, country_lookup, seed=41, max_guesses=6)

print("Target selected:", test_game.target_name)
print("Target ISO3:", test_game.target_iso3)
print("Max guesses:", test_game.max_guesses)

Target selected: Canada
Target ISO3: CAN
Max guesses: 6


# Cell 8 - Playable Worldle UI

In [ ]:
# Create a new game
game = WorldleGame(features, country_lookup, seed=41, max_guesses=6)

# Create the world map
game_map = make_map(center=(20, 0), zoom=2)

# IMPORTANT:
# Do NOT add the mystery country at the start.
# We only create the layer now, then reveal it after win/loss/give up.
target_layer = None


def reveal_target_country():
    """
    Reveal the correct country's polygon on the map.
    """
    global target_layer

    if target_layer is None:
        target_layer = add_geojson(
            game_map,
            game.target,
            style={
                "color": "#e63946",
                "fillColor": "#e63946",
                "weight": 3,
                "fillOpacity": 0.45,
            },
            name="Correct Country",
        )

        fit_map_to_geojson(game_map, game.target)


# Country choices for the searchable guess box
country_choices = sorted(
    [
        (feature["properties"]["ADMIN"], feature["properties"]["ISO_A3"])
        for feature in features
        if feature["properties"].get("ADMIN") and feature["properties"].get("ISO_A3")
    ],
    key=lambda x: x[0],
)

name_to_iso3 = {name: iso3 for name, iso3 in country_choices}

country_picker = widgets.Combobox(
    placeholder="Type a country name...",
    options=[name for name, iso3 in country_choices],
    description="Guess:",
    ensure_option=True,
    layout=widgets.Layout(width="450px"),
)

guess_button = widgets.Button(
    description="Guess",
    button_style="success",
    icon="check",
)

give_up_button = widgets.Button(
    description="Give Up",
    button_style="warning",
    icon="flag",
)

banner_output = widgets.Output()
history_output = widgets.Output()


def show_banner(html):
    """
    Clear and update the message banner.
    """
    with banner_output:
        clear_output()
        display(HTML(html))


def disable_game_buttons():
    """
    Stop the player from guessing after the game ends.
    """
    guess_button.disabled = True
    give_up_button.disabled = True
    country_picker.disabled = True


def on_guess_click(_):
    """
    Handle the Guess button.
    """
    if game.finished:
        return

    guessed_name = country_picker.value

    if guessed_name not in name_to_iso3:
        show_banner(
            """
            <div style="padding:10px; background:#fff3cd; border-radius:8px;">
                <h3>Pick a valid country from the list.</h3>
            </div>
            """
        )
        return

    guess_iso3 = name_to_iso3[guessed_name]
    result = game.submit_guess(guess_iso3)

    guess_lookup = country_lookup.get(guess_iso3, {})
    flag_path = guess_lookup.get("flag_path")

    row = render_guess_row(
        country_name=result["guess_name"],
        flag_path=flag_path,
        arrow=result["arrow"],
        distance_km=result["distance_km"],
        correct=result["correct"],
    )

    with history_output:
        display(HTML(row))

    country_picker.value = ""

    if result["message"] == "win":
        reveal_target_country()
        disable_game_buttons()

        show_banner(
            f"""
            <div style="padding:12px; background:#d1e7dd; border-radius:8px;">
                <h2>🎉 You got it!</h2>
                <p>The country was <b>{game.target_name}</b>.</p>
            </div>
            """
        )

    elif result["message"] == "loss":
        reveal_target_country()
        disable_game_buttons()

        show_banner(
            f"""
            <div style="padding:12px; background:#f8d7da; border-radius:8px;">
                <h2>Game over!</h2>
                <p>You used all {game.max_guesses} guesses.</p>
                <p>The country was <b>{game.target_name}</b>.</p>
            </div>
            """
        )

    else:
        guesses_left = game.max_guesses - len(game.guesses)

        show_banner(
            f"""
            <div style="padding:10px; background:#e2e3e5; border-radius:8px;">
                <h3>{guesses_left} guesses left.</h3>
            </div>
            """
        )


def on_give_up_click(_):
    """
    Reveal the answer and end the game.
    """
    game.finished = True
    reveal_target_country()
    disable_game_buttons()

    show_banner(
        f"""
        <div style="padding:12px; background:#fff3cd; border-radius:8px;">
            <h2>You gave up!</h2>
            <p>The country was <b>{game.target_name}</b>.</p>
        </div>
        """
    )


guess_button.on_click(on_guess_click)
give_up_button.on_click(on_give_up_click)

title = widgets.HTML(
    """
    <div style="padding:10px;">
        <h1>🌍 Worldle</h1>
        <p>Guess the mystery country. After each guess, follow the arrow, distance, color, and hot/cold emoji.</p>
        <p><b>You get 6 guesses.</b></p>
    </div>
    """
)

controls = widgets.HBox([country_picker, guess_button, give_up_button])

show_banner(
    f"""
    <div style="padding:10px; background:#e2e3e5; border-radius:8px;">
        <h3>{game.max_guesses} guesses left.</h3>
    </div>
    """
)

display(
    widgets.VBox([
        title,
        game_map,
        controls,
        banner_output,
        widgets.HTML("<h3>Guess History</h3>"),
        history_output,
    ])
)